# Legacy grid search — compatibility reference

Use `gridSearch2_nested.ipynb` for the current MacroF1-target nested evaluation.
This original notebook is preserved for comparison; its full-data fitting procedure
is not an independent generalization estimate. Ranks below are derived from current
MacroF1 only to keep the legacy ranking implementation compatible.


In [ ]:
import itertools

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import (
    spearmanr,
    kendalltau,
)

from sklearn.metrics import (
    mean_absolute_error,
)

from sklearn.preprocessing import (
    StandardScaler,
)

In [ ]:
from pathlib import Path
csv_path = Path("result1.csv")
if not csv_path.exists():
    csv_path = Path("GridSearch") / "result1.csv"
df = pd.read_csv(csv_path)
if "MacroF1" in df.columns:
    df["RealRank"] = df.groupby("Dataset")["MacroF1"].rank(ascending=False, method="min").astype(int)
print(df.head(10))
print(df.shape)

第一版用什么metric: same mean, diff mean, same abs eta, diff eta, chi2 distance.

In [ ]:
metricSeq1 = ["SameMean", "DffMean", "SameAbsEta","DiffAbsEta","Chi2Distance"]

In [ ]:
#标准化metric取值范围Z-score
def zscore(x):
    std=x.std(ddof=0)
    if np.isclose(std,0):
        return pd.Series(
            np.zeros(len(x)),
            index=x.index,
        )
    return (x-x.mean())/std

#robust 标准化:避免outlier影响
def robust(x):
    median=x.median()
    mad=np.median(np.abs(x-median))
    if np.isclose(mad, 0):
        return pd.Series(np.zeros(len(x)), index=x.index)
    return(0.6745*(x-median)/mad)

#rank按照rank标准化
def rank(x):
    ranks=x.rank(
        method="average",
        pct=True,)
    return ranks-ranks.mean()

In [ ]:
def StandardizeByDataGroup(dataframe, metricSeq, method,):
    result = dataframe.copy()
    methods = {
        "zscore":zscore,
        "robust":robust,
        "rank":rank,
    }
    standardizer = methods[method]
    standardized_metrics=[]
    for metric in metricSeq:
        outputCol = f"{metric}_scaled"
        result[outputCol]=(
            result.groupby("Dataset")[metric].transform(standardizer)
        )
        standardized_metrics.append(outputCol)

    return result, standardized_metrics

In [ ]:
zscore_df, zscore_metrics = (
    StandardizeByDataGroup(
        dataframe=df,
        metricSeq=metricSeq1,
        method="zscore",
    )
)

print(zscore_metrics)
display(
    zscore_df[
        [
            "Dataset",
            "ModelName",
            "RealRank",
        ]+zscore_metrics
    ].head(10)
)

In [ ]:
robust_df, robust_metrics = (
    StandardizeByDataGroup(
        dataframe=df,
        metricSeq=metricSeq1,
        method="robust",
    )
)

print(robust_metrics)
display(
    robust_df[
        [
            "Dataset",
            "ModelName",
            "RealRank",
        ]+robust_metrics
    ].head(10)
)

In [ ]:
rank_df, rank_metrics = (
    StandardizeByDataGroup(
        dataframe=df,
        metricSeq=metricSeq1,
        method="rank",
    )
)

print(rank_metrics)
display(
    rank_df[
        [
            "Dataset",
            "ModelName",
            "RealRank",
        ]+rank_metrics
    ].head(10)
)

### Grid Search

#### Zscore

In [ ]:
from itertools import product
from scipy.stats import kendalltau

In [ ]:
metricSeq = metricSeq1
STANDARDIZATION = "zscore"
gridStep=0.1

In [ ]:
def gridSearch(dataframe, metricSeq, step, standardizemethod):
    required = [ "Dataset", "ModelName","RealRank", *metricSeq,]
    missing = set(required) - set(dataframe.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")
    data=dataframe[required].copy()
    if data[required].isna().any().any():
        raise ValueError("There are MISSING VALUE")
    if data.duplicated(["Dataset","ModelName"]).any():
        raise ValueError("Duplicating same dataset-model pair")

    numeric_columns = ["RealRank", *metricSeq]
    data[numeric_columns] = data[numeric_columns].apply(
        pd.to_numeric,
        errors="raise",
    )
    if not np.isfinite(
        data[numeric_columns].to_numpy(dtype=float)
    ).all():
        raise ValueError("Metric or realrank has NaN/inf value")
    datasets = sorted(data["Dataset"].unique())
    if len(datasets)<3:
        raise ValueError("At least three dataset is required for grid search")
    for dataset, group in data.groupby("Dataset"):
        if len(group) < 3 or group["RealRank"].nunique() < 2:
            raise ValueError(f"{dataset}: The number of Model of real rank is not enough。")
    if not metricSeq or len(set(metricSeq)) != len(metricSeq):
        raise ValueError("metricSeq cannot been empty or having duplicate metrics")
    if not 0 < step <= 1:
        raise ValueError("range of step: (0, 1]")
    units = int(round(1 / step))
    if not np.isclose(units * step, 1.0):
        raise ValueError("step have to be like 0.1,0.05,0.2")

    #group by dataset and doing standardize separately for each group
    scaled_df, scaled_metrics = StandardizeByDataGroup(
        dataframe=data,
        metricSeq=metricSeq,
        method=standardizemethod,
    )
    if not np.isfinite(
        scaled_df[scaled_metrics].to_numpy(dtype=float)
    ).all():
        raise ValueError("there are NaN/inf in standardized result")

    # 生成不重复的系数组合 
    def compositions(total, parts):
        if parts == 1:
            yield (total,)
            return

        for value in range(total + 1):
            for rest in compositions(total - value, parts - 1):
                yield (value,) + rest

    coefficient_list = []

    for allocation in compositions(units, len(metricSeq)):
        weights = np.asarray(allocation, dtype=float) / units
        active = np.flatnonzero(weights > 0)

        # 零权重不搜索方向，避免重复
        for signs in product([-1.0, 1.0], repeat=len(active)):
            coefficients = np.zeros(len(metricSeq))
            coefficients[active] = weights[active] * np.asarray(signs)
            coefficient_list.append(coefficients)

    coefficient_grid = np.asarray(coefficient_list)
    active_counts = np.count_nonzero(coefficient_grid, axis=1)

    print("Datasets:", datasets)
    print("Metrics:", metricSeq)
    print("Unique coefficient combinations:", len(coefficient_grid))

    # ---------- 4. 预计算各候选组合在各 dataset 的排名 ----------
    # 此处只缓存计算结果。
    # 每个 LODO fold 选参数时，会严格排除 held-out dataset。
    rho_matrix = np.empty(
        (len(coefficient_grid), len(datasets))
    )
    cache = {}

    for dataset_index, dataset in enumerate(datasets):
        group = (
            scaled_df[scaled_df["Dataset"] == dataset]
            .sort_values("ModelName")
            .reset_index(drop=True)
        )

        X = group[scaled_metrics].to_numpy(dtype=float)
        scores = X @ coefficient_grid.T

        # 消除浮点计算产生的极微小伪差异
        scores = np.round(scores, 12)

        predicted_ranks = (
            pd.DataFrame(scores)
            .rank(axis=0, ascending=False, method="average")
            .to_numpy()
        )

        real_rank = group["RealRank"].to_numpy(dtype=float)

        # Spearman = 两组秩的 Pearson correlation
        true_order = (
            pd.Series(real_rank)
            .rank(method="average")
            .to_numpy()
        )
        true_centered = true_order - true_order.mean()
        pred_centered = (
            predicted_ranks - predicted_ranks.mean(axis=0)
        )

        denominator = np.sqrt(
            np.sum(true_centered ** 2)
            * np.sum(pred_centered ** 2, axis=0)
        )

        rho = np.divide(
            true_centered @ pred_centered,
            denominator,
            out=np.zeros(len(coefficient_grid)),
            where=denominator > 0,
        )

        # 所有模型同分时，Spearman原本未定义；
        # 搜索中约定记为0，表示没有排序信息。
        rho_matrix[:, dataset_index] = rho

        cache[dataset] = {
            "group": group,
            "scores": scores,
            "ranks": predicted_ranks,
            "real_rank": real_rank,
        }

    def choose_best(objective):
        maximum = objective.max()
        candidates = np.flatnonzero(
            np.isclose(objective, maximum, atol=1e-12, rtol=0)
        )

        # 平局时优先更简单的组合
        return candidates[
            np.argmin(active_counts[candidates])
        ]

    # ---------- 5. LODO：只用其他 datasets 选系数 ----------
    fold_rows = []
    prediction_parts = []
    coefficient_rows = []

    for held_out_index, held_out in enumerate(datasets):
        training_mask = np.arange(len(datasets)) != held_out_index

        training_objective = rho_matrix[
            :, training_mask
        ].mean(axis=1)

        best_index = choose_best(training_objective)
        coefficients = coefficient_grid[best_index]

        item = cache[held_out]
        predicted = item["ranks"][:, best_index]
        actual = item["real_rank"]

        tau = (
            kendalltau(actual, predicted).statistic
            if np.unique(predicted).size > 1
            else 0.0
        )

        fold_rows.append({
            "HeldOutDataset": held_out,
            "TrainSpearman": training_objective[best_index],
            "TestSpearman": rho_matrix[best_index, held_out_index],
            "TestKendall": tau,
            "TestRankMAE": np.mean(np.abs(actual - predicted)),
            "ActiveMetrics": active_counts[best_index],
        })

        predictions = item["group"][
            ["Dataset", "ModelName", "RealRank"]
        ].copy()
        predictions["PredictionScore"] = item["scores"][:, best_index]
        predictions["PredictedRank"] = predicted
        prediction_parts.append(predictions)

        for metric, coefficient in zip(metricSeq, coefficients):
            coefficient_rows.append({
                "HeldOutDataset": held_out,
                "Metric": metric,
                "Weight": abs(coefficient),
                "Direction": int(np.sign(coefficient)),
                "Coefficient": coefficient,
            })

        print(
            f"Held out: {held_out} | "
            f"Train ρ={training_objective[best_index]:.3f} | "
            f"Test ρ={rho_matrix[best_index, held_out_index]:.3f}"
        )

    fold_results = pd.DataFrame(fold_rows)

    # ---------- 6. 全数据拟合：用于后续公式和敏感性图 ----------
    # 不作为独立测试成绩
    full_objective = rho_matrix.mean(axis=1)
    final_index = choose_best(full_objective)
    final_coefficients = coefficient_grid[final_index]

    final_coefficient_df = pd.DataFrame({
        "Metric": metricSeq,
        "Weight": np.abs(final_coefficients),
        "Direction": np.sign(final_coefficients).astype(int),
        "Coefficient": final_coefficients,
    })

    return {
        "fold_results": fold_results,
        "cv_summary": fold_results[
            ["TestSpearman", "TestKendall", "TestRankMAE"]
        ].agg(["mean", "std"]),

        "predictions": pd.concat(
            prediction_parts,
            ignore_index=True,
        ),
        "fold_coefficients": pd.DataFrame(coefficient_rows),

        "final_coefficients": final_coefficient_df,
        "full_fit_spearman": full_objective[final_index],

        "scaled_df": scaled_df,
        "scaled_metrics": scaled_metrics,
        "coefficient_grid": coefficient_grid,
        "dataset_spearman": pd.DataFrame(
            rho_matrix,
            columns=datasets,
        ),
    }
    

In [ ]:
gridResult = gridSearch(
    dataframe=df,
    metricSeq=metricSeq1,
    step=gridStep,
    standardizemethod=STANDARDIZATION
)

In [ ]:
# 全部四个数据集上拟合得到的候选公式
weight_table = gridResult["final_coefficients"].copy()

weight_table["Meaning"] = weight_table["Direction"].map({
    1: "指标越大，预测得分越高",
    -1: "指标越小，预测得分越高",
    0: "不使用",
})

display(weight_table)

formula = "Score = " + " ".join(
    f"{row.Coefficient:+.2f} × Z({row.Metric})"
    for row in weight_table.itertuples()
    if row.Coefficient != 0
)

print(formula)
print("Score 越大，预测排名越靠前。")
print(
    "全数据拟合的平均 Spearman:",
    round(gridResult["full_fit_spearman"], 4),
)

# 不同留出实验选择的系数，用来观察权重稳定性
display(
    gridResult["fold_coefficients"].pivot(
        index="Metric",
        columns="HeldOutDataset",
        values="Coefficient",
    )
)

In [ ]:
def plot_grid_heatmap(
    grid_result,
    x_metric="SameMean",
    y_metric="SameAbsEta",
    label_every=2,
):
    metric_names = (
        grid_result["final_coefficients"]["Metric"].tolist()
    )

    candidates = pd.DataFrame(
        grid_result["coefficient_grid"],
        columns=metric_names,
    ).round(10)

    candidates["MeanSpearman"] = (
        grid_result["dataset_spearman"]
        .mean(axis=1)
        .to_numpy()
    )

    heatmap_data = candidates.pivot_table(
        index=y_metric,
        columns=x_metric,
        values="MeanSpearman",
        aggfunc="max",
    ).sort_index(ascending=False)

    heatmap_data = heatmap_data.reindex(
        sorted(heatmap_data.columns),
        axis=1,
    )

    # 最终选中组合的位置
    coefficients = (
        grid_result["final_coefficients"]
        .set_index("Metric")["Coefficient"]
    )

    best_col = np.flatnonzero(
        np.isclose(
            heatmap_data.columns.to_numpy(dtype=float),
            coefficients[x_metric],
        )
    )[0]

    best_row = np.flatnonzero(
        np.isclose(
            heatmap_data.index.to_numpy(dtype=float),
            coefficients[y_metric],
        )
    )[0]

    # 颜色保留全部结果，数字只显示部分位置
    annotations = np.full(
        heatmap_data.shape, "", dtype=object
    )

    for row in range(heatmap_data.shape[0]):
        for col in range(heatmap_data.shape[1]):
            value = heatmap_data.iloc[row, col]

            show_label = (
                row % label_every == 0
                and col % label_every == 0
            )

            if pd.notna(value) and show_label:
                annotations[row, col] = f"{value:.2f}"

    # 最佳位置始终显示数字
    annotations[best_row, best_col] = (
        f"{heatmap_data.iloc[best_row, best_col]:.2f}"
    )

    fig, ax = plt.subplots(figsize=(15, 13))
    ax.set_facecolor("#eeeeee")

    sns.heatmap(
        heatmap_data,
        ax=ax,
        cmap="Greens",
        vmin=-1,
        vmax=1,
        mask=heatmap_data.isna(),
        annot=annotations,
        fmt="",
        annot_kws={"fontsize": 12},
        linewidths=0.4,
        linecolor="white",
        square=True,
        cbar_kws={
            "label": "Best mean Spearman correlation",
            "shrink": 0.75,
            "pad": 0.03,
        },
    )

    # 减少坐标刻度数量
    x_positions = np.arange(
        0, heatmap_data.shape[1], label_every
    )
    y_positions = np.arange(
        0, heatmap_data.shape[0], label_every
    )

    ax.set_xticks(x_positions + 0.5)
    ax.set_xticklabels(
        [
            f"{heatmap_data.columns[i]:.1f}"
            for i in x_positions
        ],
        rotation=0,
        fontsize=13,
    )

    ax.set_yticks(y_positions + 0.5)
    ax.set_yticklabels(
        [
            f"{heatmap_data.index[i]:.1f}"
            for i in y_positions
        ],
        rotation=0,
        fontsize=13,
    )

    # 黑框标记选中组合
    ax.add_patch(
        plt.Rectangle(
            (best_col, best_row),
            1, 1,
            fill=False,
            edgecolor="black",
            linewidth=3,
        )
    )

    ax.set_xlabel(
        f"Coefficient of Z({x_metric})",
        fontsize=16,
        labelpad=14,
    )
    ax.set_ylabel(
        f"Coefficient of Z({y_metric})",
        fontsize=16,
        labelpad=14,
    )

    ax.set_title(
        "Grid Search: Coefficient Combinations",
        fontsize=19,
        fontweight="bold",
        pad=22,
    )

    colourbar = ax.collections[0].colorbar
    colourbar.ax.tick_params(labelsize=12)
    colourbar.set_label(
        "Best mean Spearman correlation",
        fontsize=14,
        labelpad=12,
    )

    fig.text(
        0.5, 0.025,
        "Full-data fit; remaining coefficients optimized separately "
        "for each cell.\n"
        "Grey: infeasible combinations. "
        "Black outline: selected combination.",
        ha="center",
        fontsize=12,
    )

    plt.tight_layout(rect=[0, 0.07, 1, 1])
    plt.show()

    return fig, heatmap_data

In [ ]:
fig, heatmap_data = plot_grid_heatmap(
    gridResult,
    x_metric="SameMean",
    y_metric="SameAbsEta",
)

In [ ]:
fig, heatmap_data = plot_grid_heatmap(
    gridResult,
    x_metric="DffMean",
    y_metric="DiffAbsEta",
)

In [ ]:
fig, heatmap_data = plot_grid_heatmap(
    gridResult,
    x_metric="DffMean",
    y_metric="Chi2Distance",
)

In [ ]:
# 查看完整的最优系数组合
best_weights = gridResult["final_coefficients"][
    ["Metric", "Coefficient"]
].copy()

display(best_weights)

# 转成字典，方便后续调用
coefficient_map = best_weights.set_index(
    "Metric"
)["Coefficient"].to_dict()

print("完整公式：")
print(
    "Score = "
    + " ".join(
        f"{coefficient:+.3f} × Z({metric})"
        for metric, coefficient in coefficient_map.items()
    )
)

print(
    "\n该组合在四个数据集上的平均 Spearman：",
    gridResult["full_fit_spearman"],
)

In [ ]:
# 使用搜索时已经标准化好的数据
final_predictions = gridResult["scaled_df"].copy()

final_predictions["PredictionScore"] = sum(
    coefficient * final_predictions[f"{metric}_scaled"]
    for metric, coefficient in coefficient_map.items()
)

# 与 grid search 中的浮点精度处理一致
final_predictions["PredictionScore"] = (
    final_predictions["PredictionScore"].round(12)
)

# 在每个数据库内部排名，分数越高越好
final_predictions["PredictedRank"] = (
    final_predictions.groupby("Dataset")["PredictionScore"]
    .rank(
        ascending=False,
        method="average",
    )
)

display(
    final_predictions[
        [
            "Dataset",
            "ModelName",
            "RealRank",
            "PredictionScore",
            "PredictedRank",
        ]
    ].sort_values(["Dataset", "PredictedRank"])
)

In [ ]:
comparison = final_predictions[
    [
        "RealRank",
        "PredictedRank",
        "ModelName",
        "Dataset",
    ]
].copy()

for dataset, group in comparison.groupby("Dataset"):
    print(f"\nDataset: {dataset}")

    display(
        group[
            ["RealRank", "PredictedRank", "ModelName"]
        ]
        .sort_values("RealRank")
        .reset_index(drop=True)
    )

#### Robust standardize

In [ ]:
STANDARDIZATION2 = "robust"

gridResult2 = gridSearch(
    dataframe=df,
    metricSeq=metricSeq1,
    step=gridStep,
    standardizemethod=STANDARDIZATION2
)

In [ ]:
# 全部四个数据集上拟合得到的候选公式
weight_table2 = gridResult2["final_coefficients"].copy()

weight_table2["Meaning"] = weight_table2["Direction"].map({
    1: "指标越大，预测得分越高",
    -1: "指标越小，预测得分越高",
    0: "不使用",
})

display(weight_table2)

formula2 = "Score = " + " ".join(
    f"{row.Coefficient:+.2f} × Z({row.Metric})"
    for row in weight_table2.itertuples()
    if row.Coefficient != 0
)

print(formula2)
print("Score 越大，预测排名越靠前。")
print(
    "全数据拟合的平均 Spearman:",
    round(gridResult2["full_fit_spearman"], 4),
)

# 不同留出实验选择的系数，用来观察权重稳定性
display(
    gridResult2["fold_coefficients"].pivot(
        index="Metric",
        columns="HeldOutDataset",
        values="Coefficient",
    )
)